# Notebook 07 — Real Club Case Study  
## Risk-Adjusted Scouting → Budget-Constrained Recruitment Decisions (Deterministic vs CVaR Robust)

**Client simulation:** *Hypothetical FC* (mid-table Big 5 league club)  
**Decision horizon:** upcoming summer window (single-period squad build)  
**Budget:** €200m (Total Cost of Ownership, TCO)  
**Squad target:** 18 players  
**Goal:** demonstrate how an analytics department would operationalise this repository’s decision system as a consulting-style report.

---

### What this notebook demonstrates (portfolio intent)

This is not a player ranking exercise.

It is a **decision system case study** that converts scouting signals into an executable squad plan under:
- performance & risk modelling  
- financial realism (TCO)  
- structural constraints (positional coverage)  
- uncertainty-aware downside protection (CVaR)

Deliverables:
- deterministic squad vs robust squad (CVaR)  
- overlap / stability diagnostics  
- downside performance diagnostics (P10 / CVaR)  
- budget allocation breakdown

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import duckdb

from scipy.optimize import milp, LinearConstraint, Bounds
from scipy import sparse

import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# 1) Club Context & Recruitment Strategy

## 1.1 Situation (simulated)
*Hypothetical FC* is a mid-table club in a Big 5 league seeking to improve competitive output without creating financial fragility.

The club wants an 18-player “core squad” plan that is:
- **budget-feasible** under a Total Cost of Ownership (TCO) framing  
- **structurally valid** (positional coverage)  
- **risk-aware** (availability / volatility proxies)  
- **downside-resilient** under performance uncertainty (CVaR)

## 1.2 Decision question
> Under a €200m TCO budget, which squad maximises sporting value while controlling risk—  
> and how does the answer change when we optimise for downside risk (CVaR)?

This notebook applies the repository pipeline:
Data → Talent → Risk → Cost → Deterministic MILP → Scenario engine → CVaR MILP → Executive reporting.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ClubPolicy:
    club_name: str = "Hypothetical FC"
    profile: str = "Mid-table Big 5 league club"
    squad_size: int = 18
    budget_eur: float = 200_000_000.0

    # Risk aversion for deterministic objective: mu = talent - lambda * risk
    lambda_risk: float = 0.60

    # Financial model (TCO)
    wage_ratio: float = 0.15       # annual wage ≈ 15% of fee proxy
    contract_years: int = 4
    discount_rate: float = 0.08

    # Robust optimisation
    alpha: float = 0.90            # CVaR confidence (tail size = 1-alpha)

    # Universe filters
    min_minutes: int = 900

policy = ClubPolicy()
policy

# 2) Market Universe (Candidate Set)

We build the modelling universe directly from the DuckDB warehouse.

**Source view:** `risk_adjusted_universe_v2`  
This view is the project’s “decision-ready” player universe (FBref performance + Transfermarkt values, engineered features, and model outputs).

We apply pragmatic operational filters:
- minimum minutes threshold (reduce small-sample noise)
- non-missing core modelling columns

In [ ]:
def find_db_path(db_name: str = "scouting.duckdb") -> Path:
    """Find db/scouting.duckdb robustly from repo_root/ or notebooks/."""
    cwd = Path.cwd()
    candidates = [cwd / "db" / db_name, cwd.parent / "db" / db_name]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(f"Could not find {db_name}. Tried: {candidates}")


DB_PATH = find_db_path()
print("Using DB at:", DB_PATH)

# close previous connection
try:
    con.close()
except:
    pass

con = duckdb.connect(str(DB_PATH), read_only=True)

In [ ]:
SQL = '''
SELECT
    player_id,
    player_name,
    season,
    age,
    season_minutes,
    position                 AS pos_primary,
    talent_score             AS mu_talent,
    risk_score               AS risk,
    market_value_in_eur      AS cost_proxy_eur,
    age_distance,
    usage_instability
FROM risk_adjusted_universe_v2
'''
df = con.execute(SQL).df()

df = df[df["season_minutes"] >= policy.min_minutes].copy()
df = df.dropna(subset=["mu_talent","risk","cost_proxy_eur","pos_primary","age_distance","usage_instability"])
df.reset_index(drop=True, inplace=True)

print("Players after filters:", len(df))
print("Positions:", sorted(df["pos_primary"].unique()))
display(df.head(3))

# 3) Financial Model — Total Cost of Ownership (TCO)

To reflect recruitment reality, we optimise under a **Total Cost of Ownership** budget rather than transfer fees alone.

We use an interpretable approximation:
- `cost_proxy_eur` = fee proxy (market value in EUR)
- annual wage ≈ `wage_ratio × cost_proxy_eur`
- wages are discounted over a `contract_years` horizon at `discount_rate`

\[
TCO_i = FeeProxy_i + \sum_{t=1}^{Y} \frac{Wage_i}{(1+r)^t}
\]

In [ ]:
def discounted_wage_pv(fee: np.ndarray, wage_ratio: float, years: int, r: float) -> np.ndarray:
    fee = np.asarray(fee, float)
    annual_wage = wage_ratio * fee
    disc = np.sum([1 / ((1 + r) ** t) for t in range(1, years + 1)])
    return annual_wage * disc

fee = df["cost_proxy_eur"].to_numpy(float)
wages_pv = discounted_wage_pv(
    fee=fee,
    wage_ratio=policy.wage_ratio,
    years=policy.contract_years,
    r=policy.discount_rate,
)
df["tco_eur"] = fee + wages_pv
df["tco_m"] = df["tco_eur"] / 1e6

print(f"Budget (TCO): €{policy.budget_eur/1e6:.1f}m")
display(df[["player_name","pos_primary","cost_proxy_eur","tco_eur","tco_m"]].head(5))

# 4) Deterministic Optimisation (MILP)

We begin with the deterministic decision metric used throughout the project:

\[
\mu_i = \text{Talent}_i - \lambda \cdot \text{Risk}_i
\]

We then solve:

\[
\max_{x \in \{0,1\}} \sum_i x_i \mu_i
\]

subject to:
- squad size = 18  
- TCO budget ≤ €200m  
- positional structure constraints (broad groups)

In [ ]:
df["mu"] = df["mu_talent"] - policy.lambda_risk * df["risk"]
mu = df["mu"].to_numpy(float)

# Convenience display
display(df[["player_name","pos_primary","age","mu_talent","risk","mu","tco_m"]].head(5))

## 4.1 Positional structure constraints (broad groups)

The universe view exposes broad positional labels:
- `Defender`, `Midfield`, `Attack` (and occasionally other labels depending on upstream mapping)

For the case study, we enforce **coverage bounds** rather than exact counts, to reflect real squad planning flexibility.

In [ ]:
# Broad position groups (consistent with Notebook 06)
POS_GROUPS = {
    "DF": ["Defender"],
    "MF": ["Midfield"],
    "FW": ["Attack"],
}

# Example bounds for an 18-player core squad
# (Adjust if your universe has different distributions)
POS_L = {"DF": 6, "MF": 6, "FW": 4}
POS_U = {"DF": 8, "MF": 8, "FW": 6}

assert sum(POS_L.values()) <= policy.squad_size <= sum(POS_U.values()), "Quota sums incompatible with squad size."

In [ ]:
def build_position_matrix(pos, pos_groups, pos_L, pos_U):
    pos = np.asarray(pos)
    group_names = list(pos_groups.keys())
    G = len(group_names)
    n = len(pos)

    A = np.zeros((G, n), dtype=float)
    for g, name in enumerate(group_names):
        A[g, :] = np.isin(pos, pos_groups[name]).astype(float)

    L = np.array([float(pos_L[name]) for name in group_names], dtype=float)
    U = np.array([float(pos_U[name]) for name in group_names], dtype=float)
    return A, L, U, group_names

In [ ]:
A_pos, L, U, group_names = build_position_matrix(
    df["pos_primary"].to_numpy(),
    POS_GROUPS,
    POS_L,
    POS_U
)
print("Eligible counts by group:", {name: int(A_pos[g,:].sum()) for g, name in enumerate(group_names)})
print("Sum L / U / squad_size:", L.sum(), U.sum(), policy.squad_size)

In [ ]:
def build_core_constraints_x(n, tco, budget, A_pos, L, U, squad_size):
    A_ub = sparse.vstack([
        sparse.csr_matrix(tco.reshape(1, -1)),
        sparse.csr_matrix(A_pos),
        sparse.csr_matrix(-A_pos),
    ], format="csr")
    b_ub = np.concatenate([np.array([budget], float), U.astype(float), (-L).astype(float)])

    A_eq = sparse.csr_matrix(np.ones((1, n), float))
    b_eq = np.array([float(squad_size)], float)
    return A_ub, b_ub, A_eq, b_eq

In [ ]:
n = len(df)
tco = df["tco_eur"].to_numpy(float)

A_ub_x, b_ub_x, A_eq_x, b_eq_x = build_core_constraints_x(
    n=n,
    tco=tco,
    budget=policy.budget_eur,
    A_pos=A_pos,
    L=L,
    U=U,
    squad_size=policy.squad_size
)
print("Core constraints built:", A_ub_x.shape, A_eq_x.shape)

In [ ]:
def solve_milp(c, A_ub=None, b_ub=None, A_eq=None, b_eq=None, bounds=None, integrality=None):
    constraints = []
    if A_ub is not None:
        constraints.append(LinearConstraint(A_ub, -np.inf, b_ub))
    if A_eq is not None:
        constraints.append(LinearConstraint(A_eq, b_eq, b_eq))
    if bounds is None:
        bounds = Bounds(-np.inf, np.inf)
    return milp(c=c, constraints=constraints, bounds=bounds, integrality=integrality)


def solve_deterministic_milp(value_mean, A_ub_x, b_ub_x, A_eq_x, b_eq_x):
    c = (-value_mean).astype(float)  # maximize => minimize negative
    bounds = Bounds(np.zeros_like(c), np.ones_like(c))
    integrality = np.ones_like(c, dtype=int)
    res = solve_milp(c, A_ub_x, b_ub_x, A_eq_x, b_eq_x, bounds=bounds, integrality=integrality)
    if not res.success:
        raise RuntimeError(res.message)
    return res.x


def build_cvar_value_milp(V_scen, p, alpha, A_ub_x, b_ub_x, A_eq_x, b_eq_x):
    # Variables: z = [x (n), eta (1), u (S)]
    # Maximise: eta - 1/(1-alpha) * sum p_s u_s
    # Minimise: -eta + 1/(1-alpha) * sum p_s u_s
    S, n = V_scen.shape
    c = np.concatenate([np.zeros(n), np.array([-1.0]), (p / (1.0 - alpha))])

    # u_s >= eta - V_s(x)  <=>  -Vx + eta - u <= 0
    A_cvar = sparse.hstack([
        sparse.csr_matrix(-V_scen),
        sparse.csr_matrix(np.ones((S, 1))),
        sparse.identity(S, format="csr") * (-1.0),
    ], format="csr")
    b_cvar = np.zeros(S)

    A_ub_lift = sparse.hstack([
        A_ub_x,
        sparse.csr_matrix((A_ub_x.shape[0], 1)),
        sparse.csr_matrix((A_ub_x.shape[0], S)),
    ], format="csr")

    A_eq_lift = sparse.hstack([
        A_eq_x,
        sparse.csr_matrix((A_eq_x.shape[0], 1)),
        sparse.csr_matrix((A_eq_x.shape[0], S)),
    ], format="csr")

    A_ub = sparse.vstack([A_cvar, A_ub_lift], format="csr")
    b_ub = np.concatenate([b_cvar, b_ub_x])

    A_eq = A_eq_lift
    b_eq = b_eq_x

    lb = np.concatenate([np.zeros(n), np.array([-np.inf]), np.zeros(S)])
    ub = np.concatenate([np.ones(n),  np.array([ np.inf]), np.full(S, np.inf)])
    integrality = np.concatenate([np.ones(n, int), np.zeros(1 + S, int)])

    return c, A_ub, b_ub, A_eq, b_eq, (lb, ub), integrality


def solve_cvar_milp(c, A_ub, b_ub, A_eq, b_eq, bounds, integrality):
    res = solve_milp(c, A_ub, b_ub, A_eq, b_eq, bounds=Bounds(bounds[0], bounds[1]), integrality=integrality)
    if not res.success:
        raise RuntimeError(res.message)
    return res.x

In [ ]:
x_det = solve_deterministic_milp(mu, A_ub_x, b_ub_x, A_eq_x, b_eq_x)
x_det = np.round(x_det).astype(int)

df_det = df.loc[x_det == 1].copy()
print("Deterministic squad size:", len(df_det))
display(df_det["pos_primary"].value_counts())

In [ ]:
def squad_table(df_squad: pd.DataFrame) -> pd.DataFrame:
    out = df_squad.copy()
    out["fee_m"] = out["cost_proxy_eur"] / 1e6
    out["tco_m"] = out["tco_eur"] / 1e6
    cols = ["player_name","pos_primary","age","mu_talent","risk","mu","fee_m","tco_m","season_minutes"]
    cols = [c for c in cols if c in out.columns]
    return out[cols].sort_values(["pos_primary","mu"], ascending=[True, False]).reset_index(drop=True)

det_table = squad_table(df_det)
det_table

# 5) Robust Optimisation (CVaR MILP)

Deterministic optimisation assumes point estimates are reliable.

To represent uncertainty (adaptation risk, role changes, sampling noise), we generate scenarios of per-player outcomes and optimise the squad’s **downside performance** using CVaR.

Interpretation:
- The robust squad can sacrifice some mean value to improve bad-case outcomes (tail protection).

In [ ]:
def build_sigma(minutes: np.ndarray, vol: np.ndarray, age_dist: np.ndarray,
                w_minutes: float = 0.35, w_vol: float = 0.45, w_age: float = 0.20,
                floor: float = 0.15, cap: float = 2.50) -> np.ndarray:
    """Interpretable uncertainty proxy σ_i."""
    minutes = np.asarray(minutes, float)
    vol = np.asarray(vol, float)
    age_dist = np.asarray(age_dist, float)

    m = np.log1p(minutes)
    m_z = (m - m.mean()) / (m.std() + 1e-9)
    u_minutes = -m_z

    v_z = (vol - vol.mean()) / (vol.std() + 1e-9)
    a_z = (age_dist - age_dist.mean()) / (age_dist.std() + 1e-9)

    sigma = w_minutes * u_minutes + w_vol * v_z + w_age * a_z
    sigma = (sigma - sigma.min()) / (sigma.max() - sigma.min() + 1e-9)
    sigma = floor + sigma * (cap - floor)
    return sigma

mu = df["mu_talent"].to_numpy(float)
risk = df["risk"].to_numpy(float)
tco = df["cost_proxy_eur"].to_numpy(float)

minutes = df["season_minutes"].to_numpy(float)
vol = df["usage_instability"].to_numpy(float)
age_dist = df["age_distance"].to_numpy(float)

value_mean = mu - policy.lambda_risk * risk
sigma = build_sigma(minutes, vol, age_dist)
df["sigma"] = sigma

print("value_mean summary:\n", pd.Series(value_mean).describe())
print("sigma summary:\n", pd.Series(sigma).describe())

In [ ]:
# ===============================
# Scenario engine configuration
# ===============================

ENGINE = "mc"        # "mc" or "regime"

S_SCEN = 400         # number of MC scenarios
SEED = 11

ALPHA = policy.alpha

# Tail stress parameters
Q_BAD = 0.25
BAD_SHIFT = -1.75

In [ ]:
@dataclass
class ScenarioSet:
    V: np.ndarray
    p: np.ndarray
    alpha: float
    scenario_names: Optional[List[str]] = None


def make_regime_scenarios(mu, sigma, risk, lam,
                          p=(0.70, 0.20, 0.10),
                          shock_levels=(0.0, -0.75, -1.75),
                          alpha=0.80) -> ScenarioSet:
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    risk = np.asarray(risk, float)

    p = np.asarray(p, float); p = p / p.sum()
    shocks = np.asarray(shock_levels, float)
    if len(p) != len(shocks):
        raise ValueError("p and shock_levels must have same length")

    T = mu[None, :] + sigma[None, :] * shocks[:, None]
    V = T - lam * risk[None, :]
    names = [f"regime_{k}_shock_{shocks[k]:+.2f}" for k in range(len(shocks))]
    return ScenarioSet(V=V, p=p, alpha=float(alpha), scenario_names=names)


def make_factor_mc_scenarios(mu, sigma, risk, lam,
                             S=150, alpha=0.80, seed=42,
                             q_bad=0.25, bad_shift=-1.75,
                             eps_scale=0.35,
                             exposure_clip=(0.5, 1.5)) -> ScenarioSet:
    rng = np.random.default_rng(seed)
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    risk = np.asarray(risk, float)
    n = len(mu)

    p = np.full(S, 1.0 / S, dtype=float)

    sigma_z = (sigma - sigma.mean()) / (sigma.std() + 1e-9)
    a = 1.0 + 0.25 * sigma_z
    a = np.clip(a, exposure_clip[0], exposure_clip[1])

    bad = rng.random(S) < q_bad
    f = rng.standard_normal(S)
    f[bad] = rng.standard_normal(bad.sum()) + bad_shift

    eps = rng.standard_normal((S, n)) * eps_scale
    Xi = f[:, None] * a[None, :] + eps

    T = mu[None, :] + sigma[None, :] * Xi
    V = T - lam * risk[None, :]
    return ScenarioSet(V=V, p=p, alpha=float(alpha))


print("Generating scenarios...")

if ENGINE == "regime":
    scen = make_regime_scenarios(
        mu, sigma, risk,
        policy.lambda_risk,
        alpha=ALPHA
    )

else:
    scen = make_factor_mc_scenarios(
        mu, sigma, risk,
        policy.lambda_risk,
        S=S_SCEN,
        alpha=ALPHA,
        seed=SEED,
        q_bad=Q_BAD,
        bad_shift=BAD_SHIFT
    )

V_scen = scen.V
p = scen.p

print("V_scen shape:", V_scen.shape, "sum(p)=", p.sum())

display(pd.DataFrame({
    "player_name": df["player_name"].head(5),
    "pos": df["pos_primary"].head(5),
    "mu": mu[:5],
    "sigma": sigma[:5],
    "v_s0": V_scen[0, :5],
}))

In [ ]:
# Uncertainty proxy σ_i (consistent with Notebook 06)
minutes = df["season_minutes"].to_numpy(float)
vol = df["usage_instability"].to_numpy(float)
age_dist = df["age_distance"].to_numpy(float)

sigma = build_sigma(minutes=minutes, vol=vol, age_dist=age_dist)
df["sigma"] = sigma

display(df[["player_name","pos_primary","mu","sigma","season_minutes","usage_instability","age_distance"]].head(5))

## 5.1 Scenario engines

We support two scenario engines (as implemented in Notebook 06):
1) **Regime shocks** (deterministic stress scenarios)  
2) **Monte Carlo factor model** (common factor + idiosyncratic noise)

For the case study, we blend both: a small set of interpretive stress tests + a Monte Carlo sample.

In [ ]:
print("Generating scenarios...")

# Regime scenarios
scen_regime = make_regime_scenarios(
    mu,
    sigma,
    risk,
    policy.lambda_risk,
    alpha=policy.alpha
)

V_regime = scen_regime.V
p_regime = scen_regime.p

# Monte Carlo scenarios
scen_mc = make_factor_mc_scenarios(
    mu,
    sigma,
    risk,
    policy.lambda_risk,
    S=S_SCEN,
    alpha=policy.alpha,
    seed=SEED,
    q_bad=Q_BAD,
    bad_shift=BAD_SHIFT
)

V_mc = scen_mc.V
p_mc = scen_mc.p

# Combine
V = np.vstack([V_regime, V_mc])
p = np.concatenate([p_regime, p_mc])
p = p / p.sum()

print("Scenario matrix shape (S x n):", V.shape)

## 5.2 Solve CVaR robust squad

We maximise:

\[
CVaR_\alpha(\text{squad value})
\]

using the linear CVaR MILP formulation (auxiliary variables for tail losses).

In [ ]:
# Build CVaR MILP components and solve (consistent with Notebook 06)
c, A_ub, b_ub, A_eq, b_eq, bounds, integrality = build_cvar_value_milp(
    V_scen=V,
    p=p,
    alpha=policy.alpha,
    A_ub_x=A_ub_x,
    b_ub_x=b_ub_x,
    A_eq_x=A_eq_x,
    b_eq_x=b_eq_x,
)

z_rob = solve_cvar_milp(c, A_ub, b_ub, A_eq, b_eq, bounds, integrality)
x_rob = np.round(z_rob[:n]).astype(int)

df_rob = df.loc[x_rob == 1].copy()
print("Robust squad size:", len(df_rob))
display(df_rob["pos_primary"].value_counts())

In [ ]:
rob_table = squad_table(df_rob)
rob_table

# 6) Deterministic vs Robust — Comparison & Trade-offs

In [ ]:
def squad_kpis(df_squad: pd.DataFrame) -> dict:
    out = {
        "players": int(len(df_squad)),
        "tco_total_m": float(df_squad["tco_eur"].sum()/1e6),
        "fee_total_m": float(df_squad["cost_proxy_eur"].sum()/1e6),
        "avg_age": float(df_squad["age"].mean()),
        "mu_sum": float(df_squad["mu"].sum()),
        "talent_sum": float(df_squad["mu_talent"].sum()),
        "risk_sum": float(df_squad["risk"].sum()),
    }

    if "sigma" in df_squad.columns:
        out["sigma_avg"] = float(df_squad["sigma"].mean())

    return out

kpi_det = squad_kpis(df_det)
kpi_rob = squad_kpis(df_rob)

comparison = pd.DataFrame([
    {"squad": "Deterministic (mean objective)", **kpi_det},
    {"squad": f"Robust CVaR (alpha={policy.alpha:.2f})", **kpi_rob},
])

comparison

In [ ]:
sel_det = set(df_det["player_id"].tolist())
sel_rob = set(df_rob["player_id"].tolist())

overlap = sel_det & sel_rob
only_det = sel_det - sel_rob
only_rob = sel_rob - sel_det

hamming = len(only_det)

print(f"Overlap: {len(overlap)} / {policy.squad_size}")
print(f"Hamming distance (changes): {hamming}")

In [ ]:
changes = pd.concat([
    df[df["player_id"].isin(list(only_det))].assign(selection="Only Deterministic"),
    df[df["player_id"].isin(list(only_rob))].assign(selection="Only Robust (CVaR)"),
], axis=0)

changes_tbl = changes[[
    "selection","player_name","pos_primary","age","mu_talent","risk","mu","sigma","tco_m","season_minutes"
]].sort_values(["selection","pos_primary","mu"], ascending=[True, True, False]).reset_index(drop=True)

changes_tbl

In [ ]:
# ---------------------------------------
# Downside diagnostics (scenario evaluation)
# ---------------------------------------

import numpy as np
import matplotlib.pyplot as plt

def cvar_left(values: np.ndarray, alpha: float) -> float:
    """
    Lower-tail CVaR at confidence alpha.
    Example: alpha=0.90 -> average of worst 10% outcomes.
    """
    v = np.sort(np.asarray(values, float))
    k = max(1, int(np.ceil((1.0 - alpha) * len(v))))
    return float(v[:k].mean())

# Ensure binary vectors (in case they are float from MILP outputs)
x_det_bin = (np.asarray(x_det) > 0.5).astype(int)
x_rob_bin = (np.asarray(x_rob) > 0.5).astype(int)

# Scenario evaluation
vals_det = V @ x_det_bin
vals_rob = V @ x_rob_bin

# Summary metrics
print("Mean det/rob:", float(np.mean(vals_det)), float(np.mean(vals_rob)))
print("P10  det/rob:", float(np.quantile(vals_det, 0.10)), float(np.quantile(vals_rob, 0.10)))
print("P05  det/rob:", float(np.quantile(vals_det, 0.05)), float(np.quantile(vals_rob, 0.05)))
print(f"CVaR det (alpha={policy.alpha:.2f}):", cvar_left(vals_det, policy.alpha))
print(f"CVaR rob (alpha={policy.alpha:.2f}):", cvar_left(vals_rob, policy.alpha))

# Distribution plot
plt.figure()
plt.hist(vals_det, bins=40, alpha=0.6, label="Deterministic")
plt.hist(vals_rob, bins=40, alpha=0.6, label=f"Robust CVaR α={policy.alpha:.2f}")
plt.axvline(np.quantile(vals_det, 0.10), linestyle="--", label="Det P10")
plt.axvline(np.quantile(vals_rob, 0.10), linestyle="--", label="Rob P10")
plt.title("Scenario Distribution of Squad Value (Deterministic vs Robust)")
plt.xlabel("Squad value (scenario)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## 6.1 Downside diagnostics (P10 / CVaR)

We evaluate both squads on the same scenario set:
- mean scenario value
- P10 (10th percentile)
- CVaR lower tail at the chosen alpha

In [ ]:
def squad_scenario_values(V: np.ndarray, x: np.ndarray) -> np.ndarray:
    return V @ x

y_det = squad_scenario_values(V, x_det)
y_rob = squad_scenario_values(V, x_rob)

diagnostics = pd.DataFrame({
    "metric": ["mean", "p10", "p05", f"cvar_left(alpha={policy.alpha:.2f})"],
    "deterministic": [
        float(np.mean(y_det)),
        float(np.quantile(y_det, 0.10)),
        float(np.quantile(y_det, 0.05)),
        float(cvar_left(y_det, policy.alpha)),
    ],
    "robust": [
        float(np.mean(y_rob)),
        float(np.quantile(y_rob, 0.10)),
        float(np.quantile(y_rob, 0.05)),
        float(cvar_left(y_rob, policy.alpha)),
    ],
})

diagnostics

In [ ]:
plt.figure()
plt.hist(y_det, bins=40, alpha=0.6, label="Deterministic")
plt.hist(y_rob, bins=40, alpha=0.6, label=f"Robust CVaR (alpha={policy.alpha:.2f})")
plt.axvline(np.quantile(y_det, 0.10), linestyle="--", label="Det P10")
plt.axvline(np.quantile(y_rob, 0.10), linestyle="--", label="Rob P10")
plt.title("Scenario Distribution of Squad Value (Deterministic vs Robust)")
plt.xlabel("Squad value (scenario)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

# 7) Budget Allocation Breakdown

We summarise how the TCO budget is allocated across broad position groups for each squad.

In [ ]:
def budget_breakdown(df_squad):

    agg_dict = {
        "player_id": "count",
        "tco_eur": "sum",
        "mu": "mean",
        "risk": "mean"
    }

    if "sigma" in df_squad.columns:
        agg_dict["sigma"] = "mean"

    return (
        df_squad
        .groupby("pos_primary")
        .agg(agg_dict)
        .rename(columns={"player_id": "players"})
        .reset_index()
    )

bb_det = budget_breakdown(df_det)
bb_rob = budget_breakdown(df_rob)

bb_det, bb_rob

In [ ]:
plt.figure()
x = np.arange(len(bb_det))
plt.bar(x - 0.2, bb_det["tco_eur"]/1e6, width=0.4, label="Deterministic")
plt.bar(x + 0.2, bb_rob["tco_eur"]/1e6, width=0.4, label=f"Robust CVaR (alpha={policy.alpha:.2f})")
plt.xticks(x, bb_det["pos_primary"])
plt.title("TCO Allocation by Position")
plt.ylabel("TCO (€m)")
plt.legend()
plt.show()

## 8) Player Replacement Analysis

To better understand how robust optimisation changes the recruitment strategy,
we examine which players are selected exclusively by each optimisation approach.

Players appearing only in the deterministic squad tend to represent
higher-variance, higher-upside profiles, while the robust squad tends to favour
more stable performance profiles.

This analysis highlights how accounting for downside risk can materially change
the recruitment shortlist.

In [ ]:
changes = pd.concat([
    df[df["player_id"].isin(list(only_det))].assign(strategy="Deterministic only"),
    df[df["player_id"].isin(list(only_rob))].assign(strategy="Robust only"),
])

changes_table = changes[[
    "strategy",
    "player_name",
    "pos_primary",
    "age",
    "mu_talent",
    "risk",
    "sigma",
    "tco_m",
    "season_minutes"
]].sort_values(["strategy","pos_primary"])

changes_table

## 9) Budget vs Downside Performance Frontier

To illustrate the strategic trade-off between financial investment and downside
performance, we compute a budget frontier.

For each budget level, the robust optimisation problem is solved again and the
resulting squad's downside performance is evaluated.

This analysis shows how increasing financial investment improves worst-case
squad outcomes.

In [ ]:
budget_grid = np.linspace(120e6, policy.budget_eur, 8)

frontier = []

for B in budget_grid:

    A_ub_B, b_ub_B, A_eq_B, b_eq_B = build_core_constraints_x(
        n=n,
        tco=tco,
        budget=float(B),
        A_pos=A_pos,
        L=L,
        U=U,
        squad_size=policy.squad_size
    )

    c, A_ub, b_ub, A_eq, b_eq, bounds, integrality = build_cvar_value_milp(
        V,
        p,
        policy.alpha,
        A_ub_B,
        b_ub_B,
        A_eq_B,
        b_eq_B
    )

    z = solve_cvar_milp(c, A_ub, b_ub, A_eq, b_eq, bounds, integrality)

    x = (z[:n] > 0.5).astype(int)

    y = V @ x

    frontier.append({
        "budget_m": B/1e6,
        "mean": np.mean(y),
        "p10": np.quantile(y,0.10),
        "cvar": cvar_left(y, policy.alpha)
    })

frontier_df = pd.DataFrame(frontier)

frontier_df

# 10) Executive Summary & Strategic Implications

## Objective

This case study demonstrates how the **Risk-Adjusted Scouting decision system** can support a football club’s recruitment strategy under:

* budget constraints (Total Cost of Ownership)
* squad structure requirements
* performance uncertainty
* downside risk control

The objective was to construct an **18-player squad under a €200M TCO budget**, comparing two decision strategies:

* **Deterministic optimisation** (maximise expected value)
* **Robust optimisation using CVaR** (protect against downside outcomes)

---

# Deterministic vs Robust Squad Construction

### Deterministic optimisation

The deterministic model maximises expected squad value:

[
\max \sum_i x_i (Talent_i - \lambda Risk_i)
]

This approach prioritises players with the highest expected performance signal without explicitly accounting for scenario uncertainty.

Results show that the deterministic squad achieves:

* **Higher expected value**
* Greater concentration in high-upside profiles
* Larger exposure to downside scenarios

---

### Robust optimisation (CVaR)

The robust model instead maximises **Conditional Value at Risk (CVaR)** across simulated performance scenarios.

This objective prioritises squads that perform **better in adverse outcomes**, reducing the probability of severe underperformance.

The resulting squad shows:

* Slightly lower expected value
* Substantially improved downside stability
* A more diversified risk structure across positions

---

# Scenario Performance Comparison

| Metric        | Deterministic | Robust CVaR |
| ------------- | ------------- | ----------- |
| Mean value    | **16.52**     | 15.13       |
| P10           | -16.66        | **2.61**    |
| P05           | -30.70        | **-3.07**   |
| CVaR (α=0.90) | -32.85        | **-3.40**   |

Key insight:

* The deterministic squad delivers a higher mean outcome.
* However, the **robust squad dramatically improves worst-case performance**.

For example:

[
CVaR_{det} = -32.85 \quad vs \quad CVaR_{rob} = -3.40
]

This represents a **large reduction in downside risk exposure**.

---

# Squad Stability Analysis

The two optimisation strategies produce materially different recruitment outcomes.

* Squad size: **18 players**
* Overlap between squads: **9 players**
* **Hamming distance: 9 players replaced**

This indicates that **half of the squad composition changes when downside risk is incorporated**.

This is a key insight:

> Risk-aware optimisation does not simply reorder players — it can fundamentally alter the recruitment shortlist.

---

# Player Replacement Insights

Examining the players selected exclusively by each strategy provides additional insight into how the optimisation framework behaves.

Players selected **only by the deterministic model** tend to exhibit:

* higher upside potential
* greater statistical volatility
* more uncertain performance profiles

By contrast, players selected **only by the robust model** typically show:

* higher availability (minutes played)
* lower volatility in performance indicators
* more stable contribution profiles

This confirms that **robust optimisation implicitly favours stability over upside risk**.

---

# Budget Allocation Strategy

The financial allocation across positions also differs significantly.

**Deterministic squad**

* Strong concentration of investment in attacking profiles
* Higher exposure to high-variance offensive players

**Robust squad**

* More balanced spending across positions
* Greater investment in midfield stability
* Reduced concentration of financial risk in attacking roles

This pattern resembles **portfolio diversification strategies in financial asset allocation**.

---

# Budget vs Downside Performance Frontier

The budget frontier analysis illustrates how increasing investment affects downside squad performance.

| Budget (€M) | Mean  | P10   | CVaR  |
| ----------- | ----- | ----- | ----- |
| 120         | 14.55 | -0.02 | -6.58 |
| 131         | 15.03 | 2.27  | -3.56 |
| 142         | 16.12 | 4.33  | -1.00 |
| 154         | 17.49 | 5.99  | 0.83  |
| 165         | 18.27 | 7.51  | 2.49  |
| 177         | 20.04 | 9.52  | 3.74  |
| 188         | 20.88 | 10.38 | 5.55  |
| 200         | 21.66 | 11.89 | 7.21  |

The results show a clear pattern:

* Increasing squad investment **systematically improves downside outcomes**
* CVaR increases from **-6.6 at €120M** to **+7.2 at €200M**

This demonstrates how the framework can support **strategic financial planning**, allowing decision-makers to quantify the relationship between budget and recruitment risk.

---

# Practical Implications for Recruitment Strategy

The analysis highlights two distinct recruitment philosophies.

### Deterministic strategy

Best suited for clubs that:

* can tolerate performance volatility
* prioritise upside potential
* have sufficient squad depth to absorb recruitment failures

### Robust strategy

More appropriate for clubs that:

* operate under financial constraints
* require stable performance
* want to minimise recruitment risk

For a **mid-table Big 5 club**, the robust approach may provide a more sustainable recruitment framework.

---

# Key Takeaway

This case study illustrates how football analytics can evolve from descriptive player evaluation toward **decision-oriented recruitment modelling**.

By integrating:

* performance analytics
* risk modelling
* financial constraints
* optimisation under uncertainty

clubs can construct **risk-aware squads rather than simply ranking players by talent**.

---

# Final Remark

While this study uses public data and simplified financial assumptions, the methodology can be extended with:

* injury probability models
* multi-season squad planning
* contract and wage dynamics
* richer performance metrics

Such extensions would move the framework closer to a **fully integrated football recruitment decision system**.

---

> This notebook concludes the end-to-end pipeline from data ingestion to robust squad construction under uncertainty.
